# Milestone 5: Statistical Analytics
## Kenyan Supermarket Synthetic Dataset
This notebook compares average record-level spend across payment methods and across sales records with and without a promotion ID. It also checks distribution characteristics and documents important limitations.

**Important:** These are observational comparisons. A significant difference does not prove that promotions caused higher sales. Repeated transactions from the same customers may violate the independence assumption.

## 1. Load the cleaned datasets
The notebook searches common project folders and your Downloads folder. If the CSV files are stored elsewhere, set `DATA_DIR` below.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats

# If needed, set this to the folder containing your cleaned CSV files.
# Example: Path("data/processed")
DATA_DIR = None

project_root = Path.cwd()
candidate_dirs = []
if DATA_DIR is not None:
    candidate_dirs.append(Path(DATA_DIR))
candidate_dirs.extend([
    project_root / "data" / "processed",
    project_root / "data",
    project_root,
    Path.home() / "Downloads",
])

def find_csv(filename):
    for folder in candidate_dirs:
        path = folder / filename
        if path.exists():
            return path
    checked = "\n".join(str(folder / filename) for folder in candidate_dirs)
    raise FileNotFoundError(
        f"Could not find {filename}. Checked:\n{checked}\n"
        "Set DATA_DIR to the folder containing your cleaned CSV files."
    )

file_names = {
    "sales": "sales_validated.csv",
    "products": "products_cleaned.csv",
    "customers": "customers_cleaned.csv",
    "stores": "stores_cleaned.csv",
    "promotions": "promotions_cleaned.csv",
    "calendar": "calendar_cleaned.csv",
}

datasets = {name: pd.read_csv(find_csv(filename)) for name, filename in file_names.items()}
sales = datasets["sales"]

for name, df in datasets.items():
    print(f"{name.title()}: {df.shape[0]:,} rows, {df.shape[1]} columns")

## 2. Inspect sales data quality

In [ ]:
print("Sales columns:")
print(sales.columns.tolist())

key_columns = [
    col for col in ["payment_method", "quantity", "selling_price_kes", "transaction_date"]
    if col in sales.columns
]
print("\nMissing values in key columns:")
print(sales[key_columns].isna().sum())

print("\nPayment method counts:")
print(sales["payment_method"].value_counts(dropna=False))

sales["transaction_date"] = pd.to_datetime(sales["transaction_date"], errors="coerce")
print("\nSales date range:")
print(sales["transaction_date"].min())
print(sales["transaction_date"].max())

## 3. Prepare record-level spend
This analysis calculates `spend_kes = quantity × selling_price_kes`. This assumes `selling_price_kes` is a per-unit price. If it already represents the total value for the transaction line, use `selling_price_kes` directly instead.

In [ ]:
required = {"quantity", "selling_price_kes", "payment_method"}
missing = required - set(sales.columns)
if missing:
    raise KeyError(f"Required columns are missing: {sorted(missing)}")

test_data = sales.copy()
test_data["quantity"] = pd.to_numeric(test_data["quantity"], errors="coerce")
test_data["selling_price_kes"] = pd.to_numeric(test_data["selling_price_kes"], errors="coerce")
test_data["spend_kes"] = test_data["quantity"] * test_data["selling_price_kes"]
test_data = test_data.dropna(subset=["spend_kes", "payment_method"])
test_data["payment_group"] = np.where(
    test_data["payment_method"].eq("M-Pesa"), "M-Pesa", "Other payment methods"
)

payment_summary = test_data.groupby("payment_group")["spend_kes"].agg(
    ["count", "mean", "median", "std", "min", "max"]
).round(2)
display(payment_summary)

## 4. Hypothesis test 1: M-Pesa versus other payment methods
- Null hypothesis (H₀): Mean spend is equal for the two groups.
- Alternative hypothesis (H₁): Mean spend differs between the two groups.
- Significance level: 0.05.

In [ ]:
mpesa = test_data.loc[test_data["payment_group"] == "M-Pesa", "spend_kes"]
other = test_data.loc[test_data["payment_group"] == "Other payment methods", "spend_kes"]

t_stat, p_value = stats.ttest_ind(mpesa, other, equal_var=False, nan_policy="omit")
difference = mpesa.mean() - other.mean()

# Welch-Satterthwaite confidence interval for the difference in means
a = mpesa.var(ddof=1) / len(mpesa)
b = other.var(ddof=1) / len(other)
se = np.sqrt(a + b)
df = (a + b) ** 2 / (a**2 / (len(mpesa) - 1) + b**2 / (len(other) - 1))
critical = stats.t.ppf(0.975, df)
ci_lower, ci_upper = difference - critical * se, difference + critical * se

print(f"M-Pesa mean spend: KES {mpesa.mean():,.2f}")
print(f"Other methods mean spend: KES {other.mean():,.2f}")
print(f"Mean difference (M-Pesa - other): KES {difference:,.2f}")
print(f"Welch t-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"95% confidence interval: KES {ci_lower:,.2f} to KES {ci_upper:,.2f}")

if p_value < 0.05:
    print("Decision: Reject the null hypothesis at the 5% significance level.")
else:
    print("Decision: Fail to reject the null hypothesis at the 5% significance level.")

### Interpretation
Previously observed results showed mean spend of about KES 1,395.08 for M-Pesa records and KES 1,388.15 for other payment methods. The p-value was approximately 0.0984, and the 95% confidence interval was approximately KES -1.29 to KES 15.15. Since the p-value was above 0.05 and the interval included zero, there was insufficient evidence of a difference at the 5% significance level. Run the code above to recalculate results from the current CSV files.

## 5. Examine promotion IDs and discount rates

In [ ]:
if "promotion_id" not in sales.columns or "discount_rate" not in sales.columns:
    raise KeyError("The sales data must include promotion_id and discount_rate.")

print("Promotion ID missing/present counts (True means missing):")
print(sales["promotion_id"].isna().value_counts())

print("\nDiscount rate counts:")
print(sales["discount_rate"].value_counts(dropna=False).sort_index())

promotion_group = sales["promotion_id"].notna().map({
    True: "Promotion ID present", False: "Promotion ID missing"
})
print("\nDiscount rate summary by promotion ID status:")
display(
    sales.assign(promotion_group=promotion_group)
         .groupby("promotion_group")["discount_rate"]
         .agg(["count", "mean", "min", "max"])
         .round(4)
)

## 6. Hypothesis test 2: with versus without a promotion ID
- Null hypothesis (H₀): Mean record-level sales value is equal in both groups.
- Alternative hypothesis (H₁): Mean record-level sales value differs between the groups.
This is a test of association, not causation.

In [ ]:
promo_data = test_data.copy()
promo_data["has_promotion"] = sales.loc[promo_data.index, "promotion_id"].notna()
promoted = promo_data.loc[promo_data["has_promotion"], "spend_kes"].dropna()
not_promoted = promo_data.loc[~promo_data["has_promotion"], "spend_kes"].dropna()

t_stat_promo, p_value_promo = stats.ttest_ind(
    promoted, not_promoted, equal_var=False, nan_policy="omit"
)
difference_promo = promoted.mean() - not_promoted.mean()

a = promoted.var(ddof=1) / len(promoted)
b = not_promoted.var(ddof=1) / len(not_promoted)
se_promo = np.sqrt(a + b)
df_promo = (a + b) ** 2 / (a**2 / (len(promoted) - 1) + b**2 / (len(not_promoted) - 1))
critical_promo = stats.t.ppf(0.975, df_promo)
ci_lower_promo = difference_promo - critical_promo * se_promo
ci_upper_promo = difference_promo + critical_promo * se_promo

print(f"With promotion ID: n={len(promoted):,}, mean=KES {promoted.mean():,.2f}")
print(f"Without promotion ID: n={len(not_promoted):,}, mean=KES {not_promoted.mean():,.2f}")
print(f"Mean difference (with - without): KES {difference_promo:,.2f}")
print(f"Welch t-statistic: {t_stat_promo:.4f}")
print(f"P-value: {p_value_promo:.6f}")
print(f"95% confidence interval: KES {ci_lower_promo:,.2f} to KES {ci_upper_promo:,.2f}")

if p_value_promo < 0.05:
    print("Decision: Reject the null hypothesis at the 5% significance level.")
else:
    print("Decision: Fail to reject the null hypothesis at the 5% significance level.")